In [0]:
-- VP Q01 Daily energy
SELECT
    session_date,
    completed_sessions,
    energy_kwh,
    billed_amount  AS total_billed,
    amount_paid    AS total_paid
FROM vp_rp20040126.gold.agg_daily_station
ORDER BY session_date , completed_sessions;

-- VP Q02 Station utilisation
SELECT
    station_id,
    city,
    SUM(completed_sessions)       AS completed_sessions,
    SUM(energy_kwh)               AS energy_kwh,
    ROUND(AVG(utilisation_pct), 2) AS utilisation_pct
FROM vp_rp20040126.gold.agg_daily_station
GROUP BY station_id, city
ORDER BY utilisation_pct DESC;

-- VP Q03 Billing by plan
SELECT
    plan_at_session,
    COUNT(CASE WHEN status = 'COMPLETED'     THEN 1 END)   AS completed_sessions,
    COUNT(CASE WHEN billing_status = 'MISMATCH' THEN 1 END)   AS mismatch_sessions,
    SUM(CASE WHEN status = 'COMPLETED'
             THEN billed_amount - amount_paid ELSE 0 END)      AS revenue_leakage
FROM vp_rp20040126.gold.fact_sessions
WHERE plan_at_session IN ('BASIC', 'PRO', 'FLEET', 'GUEST')
GROUP BY plan_at_session
ORDER BY plan_at_session;

-- VP Q04 Charger failures
SELECT
    charger_id,
    COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)  AS completed_sessions,
    COUNT(CASE WHEN status = 'FAILED'    THEN 1 END)  AS failed_sessions,
    ROUND(
        COUNT(CASE WHEN status = 'FAILED' THEN 1 END) * 100.0 /
        NULLIF(COUNT(CASE WHEN status IN ('COMPLETED', 'FAILED') THEN 1 END), 0), 2
    ) AS failure_rate_pct
FROM vp_rp20040126.gold.fact_sessions
GROUP BY charger_id
HAVING COUNT(CASE WHEN status IN ('COMPLETED', 'FAILED') THEN 1 END) >= 10
ORDER BY failure_rate_pct DESC, charger_id
LIMIT 10;

-- VP Q05 City scorecard
SELECT
    COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END)   AS completed_sessions,
    SUM(CASE WHEN status = 'COMPLETED' THEN energy_kwh ELSE 0 END)  AS energy_kwh,
    SUM(CASE WHEN status = 'COMPLETED' THEN billed_amount ELSE 0 END) AS total_billed,
    ROUND(
        COUNT(CASE WHEN status = 'COMPLETED' THEN 1 END) * 100.0 /
        NULLIF(COUNT(*), 0), 2
    ) AS success_rate_pct
FROM vp_rp20040126.gold.fact_sessions
WHERE station_id IN (
        SELECT station_id FROM vp_rp20040126.gold.dim_station WHERE city = '{{city}}'
    )
  AND date_key >= int(replace('{{start_date}}', '-', ''))
  AND date_key <= int(replace('{{end_date}}', '-', ''));